# Clasificación del desempeño en objetos interactuados

Este notebook crea una variable objetivo de tres clases a partir de `ObjetosInteractuadosCorrectamente` (conteo de 0 a 9): **bajo** (0–3), **medio** (4–6) y **alto** (7–9). Las bandas son fáciles de interpretar y dejan tamaños de clase razonablemente equilibrados en las 117 sesiones fuente actuales.

El análisis usa la base aumentada de 6.000 filas. Como las sintéticas derivan de sesiones reales, la validación reserva grupos completos por sesión fuente y la prueba final incluye solo sesiones reales. El modelo predice el nivel usando otras variables de la sesión, como tiempos, precisión, aciertos, edad y tipo de escena. Se excluyen identificadores personales y variables que revelarían directamente el objetivo. Esto es un ejercicio exploratorio, no un diagnóstico clínico.


## 1. Librerías y carga de datos
Ejecuta el notebook desde la carpeta del proyecto. Si hace falta, ajusta la ruta al CSV.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import GroupShuffleSplit, GroupKFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (classification_report, ConfusionMatrixDisplay,
                             balanced_accuracy_score, f1_score)

RUTA = Path("neuroflex_dataset_augmented.csv")
if not RUTA.exists():
    RUTA = Path("../neuroflex_dataset_augmented.csv")
df = pd.read_csv(RUTA)
print(f"Filas: {len(df)} | Columnas: {df.shape[1]}")
print("Filas reales:", (df["source_session_id"].fillna("").astype(str).str.strip() == "").sum())
df.head()


## 2. Construcción de la variable objetivo
Los cortes 0–3, 4–6 y 7–9 mantienen significado en la escala original y, en la muestra disponible, evitan una clase muy pequeña. Se reporta la distribución para revisar el balance.

In [ ]:
OBJETIVO_ORIGINAL = "ObjetosInteractuadosCorrectamente"

# Convierte posibles valores mal formateados a faltantes y descarta filas sin etiqueta.
df[OBJETIVO_ORIGINAL] = pd.to_numeric(df[OBJETIVO_ORIGINAL], errors="coerce")
df = df.dropna(subset=[OBJETIVO_ORIGINAL]).copy()

# La medida tiene rango observado 0–9; se recortan valores fuera de rango por seguridad.
conteo = df[OBJETIVO_ORIGINAL].clip(0, 9)
df["nivel_desempeno"] = pd.cut(
    conteo, bins=[-1, 3, 6, 9], labels=["bajo", "medio", "alto"]
)

orden = ["bajo", "medio", "alto"]
distribucion = df["nivel_desempeno"].value_counts().reindex(orden)
resumen = pd.DataFrame({"sesiones": distribucion, "porcentaje": (distribucion / len(df) * 100).round(1)})
resumen


In [ ]:
ax = sns.barplot(x=resumen.index, y=resumen["sesiones"], hue=resumen.index, palette="viridis", legend=False)
ax.set(title="Distribución de la variable objetivo", xlabel="Nivel de desempeño", ylabel="Número de sesiones")
plt.show()


## 3. Preparación de predictores
Se excluyen el conteo objetivo y campos que identifican sesiones/personas o describen la fecha. `Precision` y `CantAciertasTotales` sí se conservan como posibles señales predictoras; revisa su definición en el proceso de captura para confirmar que estarían disponibles al momento de hacer una predicción real.

In [ ]:
columnas_excluir = [
    OBJETIVO_ORIGINAL, "nivel_desempeno", "timestamp", "session_id", "date",
    "name", "rut", "patient_updated_at", "source_session_id"
]
# Las filas originales tienen source_session_id vacío; las sintéticas apuntan a su sesión fuente.
es_real = df["source_session_id"].fillna("").astype(str).str.strip().eq("")
grupos = df["source_session_id"].where(~es_real, df["session_id"]).astype(str)
X = df.drop(columns=[c for c in columnas_excluir if c in df.columns]).copy()
y = df["nivel_desempeno"].astype(str)

# Eliminar columnas completamente vacías y columnas constantes.
X = X.dropna(axis=1, how="all")
X = X.loc[:, X.nunique(dropna=True) > 1]

# Convertir columnas numéricas que el CSV pudiera haber leído como texto.
for col in X.columns:
    if X[col].dtype == "object":
        convertido = pd.to_numeric(X[col], errors="coerce")
        if convertido.notna().mean() >= 0.9:
            X[col] = convertido

numericas = X.select_dtypes(include=np.number).columns.tolist()
categoricas = [c for c in X.columns if c not in numericas]
print("Predictores:", X.columns.tolist())
print("Numéricas:", numericas)
print("Categóricas:", categoricas)


## 4. Separación y modelos
Se reservan grupos completos de sesiones fuente. El entrenamiento usa las filas aumentadas de esos grupos y la prueba contiene solo sesiones reales no vistas. El preprocesamiento se ajusta dentro de cada pipeline.

In [ ]:
# Se separan grupos de sesiones fuente. El test contiene solo observaciones reales;
# el entrenamiento puede usar filas reales y sintéticas de los grupos de entrenamiento.
ids_reales = np.flatnonzero(es_real.to_numpy())
X_reales = X.iloc[ids_reales]
y_reales = y.iloc[ids_reales]
grupos_reales = grupos.iloc[ids_reales]
splitter = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
idx_train_reales, idx_test_reales = next(splitter.split(X_reales, y_reales, groups=grupos_reales))
grupos_train = set(grupos_reales.iloc[idx_train_reales])
grupos_test = set(grupos_reales.iloc[idx_test_reales])
mask_train = grupos.isin(grupos_train)
mask_test = es_real & grupos.isin(grupos_test)
X_train, y_train, grupos_train_filas = X.loc[mask_train], y.loc[mask_train], grupos.loc[mask_train]
X_test, y_test = X.loc[mask_test], y.loc[mask_test]
print(f"Entrenamiento: {len(X_train)} filas (incluye sintéticas)")
print(f"Prueba: {len(X_test)} sesiones reales")
print("Distribución test:")
print(y_test.value_counts().reindex(orden))

prep_lr = ColumnTransformer([
    ("num", Pipeline([("imputer", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), numericas),
    ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                       ("onehot", OneHotEncoder(handle_unknown="ignore"))]), categoricas),
])
prep_tree = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), numericas),
    ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                       ("onehot", OneHotEncoder(handle_unknown="ignore"))]), categoricas),
])

modelos = {
    "Regresión logística": Pipeline([("prep", prep_lr), ("modelo", LogisticRegression(max_iter=2000, class_weight="balanced"))]),
    "Random Forest": Pipeline([("prep", prep_tree), ("modelo", RandomForestClassifier(n_estimators=400, min_samples_leaf=2, class_weight="balanced", random_state=42))]),
}


## 5. Validación cruzada en entrenamiento
La selección se basa en F1 macro, que da el mismo peso a bajo, medio y alto. Con esta muestra pequeña, interpreta el resultado con cautela.

In [ ]:
cv = GroupKFold(n_splits=5)
filas = []
for nombre, pipe in modelos.items():
    scores = cross_validate(pipe, X_train, y_train, groups=grupos_train_filas, cv=cv, scoring={
        "f1_macro": "f1_macro", "balanced_accuracy": "balanced_accuracy"
    })
    filas.append({"modelo": nombre, "F1 macro CV": scores["test_f1_macro"].mean(),
                  "DE F1": scores["test_f1_macro"].std(),
                  "balanced accuracy CV": scores["test_balanced_accuracy"].mean()})
resultados_cv = pd.DataFrame(filas).sort_values("F1 macro CV", ascending=False)
resultados_cv


In [ ]:
nombre_elegido = resultados_cv.iloc[0]["modelo"]
mejor_modelo = modelos[nombre_elegido]
mejor_modelo.fit(X_train, y_train)
pred = mejor_modelo.predict(X_test)
print("Modelo elegido:", nombre_elegido)
print("Balanced accuracy test:", round(balanced_accuracy_score(y_test, pred), 3))
print("F1 macro test:", round(f1_score(y_test, pred, average="macro"), 3))
print(classification_report(y_test, pred, labels=orden, zero_division=0))


In [ ]:
ConfusionMatrixDisplay.from_predictions(y_test, pred, labels=orden, display_labels=orden, cmap="Blues", values_format="d")
plt.title("Matriz de confusión — conjunto de prueba")
plt.show()


## 6. Interpretación y próximos pasos

- La matriz de confusión ayuda a ver si el modelo confunde niveles cercanos, especialmente `medio` con `bajo` o `alto`.
- Aunque hay 6.000 filas, las sesiones fuente reales son 117, las métricas pueden variar mucho según la partición. Reporta F1 macro junto con la matriz de confusión, y evita concluir que el modelo está listo para uso clínico.
- Confirma que los predictores estén disponibles antes de conocer el resultado de objetos interactuados. Si `Precision` o `CantAciertasTotales` se calculan usando ese mismo resultado, elimínalos para evitar fuga de información.
- Para una evaluación más sólida, reúne más sesiones y, si hay varias sesiones por persona, divide entrenamiento y prueba por participante, no por fila.
